# Bab 24: Rantai Markov dan Hidden Markov Model

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab24/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab24")
print("Folder kerja:", os.getcwd())

## Listing 24.1: Rantai Markov mini dan keadaan serap

`kode/bab24/markov_mini.py`

In [ ]:
import numpy as np
import pandas as pd

sesi = ["M K S", "M M K S", "M K M K S",
        "M M X", "M M K X", "M X"]
keadaan = ["M", "K", "S", "X"]
hitung = pd.DataFrame(0, index=keadaan, columns=keadaan)
for s in sesi:
    x = s.split()
    for a, b in zip(x[:-1], x[1:]):
        hitung.loc[a, b] += 1
print("banyak transisi:\n", hitung.loc[["M", "K"]])

P = hitung.loc[["M", "K"]].div(hitung.loc[["M", "K"]].sum(1),
                                axis=0)
print("\npeluang transisi:\n", P.round(2))

Q = P[["M", "K"]].to_numpy()
R = P[["S", "X"]].to_numpy()
N = np.linalg.inv(np.eye(2) - Q)
print("\nmatriks fundamental N:\n", N.round(3))
print("harapan langkah sampai selesai:", N.sum(1).round(2))
B = N @ R
print("peluang berakhir di S / X:")
for i, k in enumerate(["M", "K"]):
    print(f"  dari {k}: S = {B[i, 0]:.2f}, X = {B[i, 1]:.2f}")

P_penuh = np.zeros((4, 4))
P_penuh[:2] = P.to_numpy()
P_penuh[2, 2] = P_penuh[3, 3] = 1
P2 = np.linalg.matrix_power(P_penuh, 2)
print("\nP(M ke S dalam tepat 2 langkah) =", round(P2[0, 2], 2))
P50 = np.linalg.matrix_power(P_penuh, 50)
print("P^50 baris M:", P50[0].round(3))

## Listing 24.2: Forward, backward, dan Viterbi pada HMM mini

`kode/bab24/hmm_mini.py`

In [ ]:
import numpy as np
from hmmlearn.hmm import CategoricalHMM

pi = np.array([0.6, 0.4])                    # J = jelajah, N = niat
A = np.array([[0.7, 0.3],
              [0.2, 0.8]])
B = np.array([[0.6, 0.3, 0.1],               # simbol: m, d, k
              [0.2, 0.3, 0.5]])
o = [0, 1, 2]                                # m d k

alfa = np.zeros((3, 2))
alfa[0] = pi * B[:, o[0]]
for t in range(1, 3):
    alfa[t] = (alfa[t - 1] @ A) * B[:, o[t]]
print("alfa:\n", alfa.round(5))
print("P(O) =", round(alfa[-1].sum(), 5))
print("P(N | m d k) =", round(alfa[-1, 1] / alfa[-1].sum(), 3))

beta = np.ones((3, 2))
for t in (1, 0):
    beta[t] = A @ (B[:, o[t + 1]] * beta[t + 1])
gamma = alfa * beta / alfa[-1].sum()
print("beta:\n", beta.round(4))
print("gamma (posterior J, N):\n", gamma.round(3))

delta = np.zeros((3, 2)); asal = np.zeros((3, 2), int)
delta[0] = pi * B[:, o[0]]
for t in range(1, 3):
    calon = delta[t - 1][:, None] * A
    asal[t] = calon.argmax(0)
    delta[t] = calon.max(0) * B[:, o[t]]
jalur = [int(delta[-1].argmax())]
for t in (2, 1):
    jalur.insert(0, int(asal[t, jalur[0]]))
print("delta:\n", delta.round(5))
print("jalur Viterbi:", "".join("JN"[j] for j in jalur),
      " peluang", round(delta[-1].max(), 5))

m = CategoricalHMM(n_components=2)
m.startprob_, m.transmat_, m.emissionprob_ = pi, A, B
X = np.array(o).reshape(-1, 1)
print("hmmlearn: log P(O) =", round(m.score(X), 4),
      "=", round(np.log(alfa[-1].sum()), 4))
logp, jl = m.decode(X, algorithm="viterbi")
print("hmmlearn Viterbi:", jl, round(np.exp(logp), 5))

## Listing 24.3: Baum–Welch dari nol dan dengan hmmlearn

`kode/bab24/baum_welch.py`

In [ ]:
import numpy as np
from hmmlearn.hmm import CategoricalHMM

pi0 = np.array([0.6, 0.4])
A0 = np.array([[0.7, 0.3], [0.2, 0.8]])
B0 = np.array([[0.6, 0.3, 0.1], [0.2, 0.3, 0.5]])
data = [[0, 1, 2], [0, 0, 1], [1, 2, 2], [0, 1, 0]]  # mdk mmd dkk mdm


def e_step(o, pi, A, B):
    T = len(o)
    a = np.zeros((T, 2)); b = np.ones((T, 2))
    a[0] = pi * B[:, o[0]]
    for t in range(1, T):
        a[t] = (a[t - 1] @ A) * B[:, o[t]]
    for t in range(T - 2, -1, -1):
        b[t] = A @ (B[:, o[t + 1]] * b[t + 1])
    p = a[-1].sum()
    g = a * b / p
    xi = [a[t][:, None] * A * (B[:, o[t + 1]] * b[t + 1]) / p
          for t in range(T - 1)]
    return g, np.sum(xi, 0), np.log(p)


def m_step(data, pi, A, B):
    G0, Xi, Gt, Bn, ll = 0, 0, 0, np.zeros_like(B), 0
    for o in data:
        g, xi, lp = e_step(o, pi, A, B)
        G0 = G0 + g[0]; Xi = Xi + xi; Gt = Gt + g[:-1].sum(0)
        for t, s in enumerate(o):
            Bn[:, s] += g[t]
        ll += lp
    return (G0 / len(data), Xi / Gt[:, None],
            Bn / Bn.sum(1, keepdims=True), ll)


pi, A, B = pi0, A0, B0
for it in range(1, 31):
    pi, A, B, ll = m_step(data, pi, A, B)
    if it in (1, 2, 3, 10, 30):
        print(f"iterasi {it:2d}: log-likelihood sebelum = {ll:.4f}")
    if it == 1:
        print("  pi =", pi.round(3), " A =", A.round(3).tolist())
        print("  B  =", B.round(3).tolist())
print("akhir: A =", A.round(3).tolist())
print("       B =", B.round(3).tolist())

m = CategoricalHMM(2, n_iter=30, tol=0, init_params="",
                   params="ste")
m.startprob_, m.transmat_, m.emissionprob_ = pi0, A0, B0
X = np.concatenate(data).reshape(-1, 1)
m.fit(X, lengths=[len(o) for o in data])
print("hmmlearn A =", m.transmat_.round(3).tolist())
print("hmmlearn B =", m.emissionprob_.round(3).tolist())

## Listing 24.4: PrefixSpan sederhana dan pola sesi aplikasi

`kode/bab24/pola_sekuensial.py`

In [ ]:
from collections import Counter
import pandas as pd


def prefixspan(db, min_sup, prefix=(), hasil=None):
    hasil = {} if hasil is None else hasil
    dukung = Counter()
    for s in db:
        dukung.update(set(s))
    for item, n in sorted(dukung.items()):
        if n < min_sup:
            continue
        pola = prefix + (item,)
        hasil[pola] = n
        proyeksi = [s[s.index(item) + 1:] for s in db if item in s]
        prefixspan([p for p in proyeksi if p], min_sup, pola, hasil)
    return hasil


mini = ["B P M K", "B M D K", "P M D", "B M K"]
db = [s.split() for s in mini]
for pola, n in prefixspan(db, 3).items():
    print(" -> ".join(pola), f"(dukungan {n}/4)")

d = pd.read_csv("kode/data/sesi_aplikasi.csv")
seq = d.groupby("id_sesi").kejadian.apply(list)
beli = seq.map(lambda s: s[-1] == "selesai")
inti = seq.map(lambda s: [x for x in s[:-1] if x != "beranda"])
db = [tuple(dict.fromkeys(s)) for s in inti]     # buang ulangan

pola = prefixspan([list(s) for s in db], min_sup=250)
print(f"\n{len(pola)} pola dengan dukungan >= 250 sesi (5%)")


def memuat(s, p):
    it = iter(s)
    return all(x in it for x in p)


baris = []
for p, n in pola.items():
    if len(p) >= 2 and "bayar" not in p:
        cocok = [memuat(s, p) for s in db]
        baris.append((" > ".join(p), n, beli[cocok].mean()))
tab = pd.DataFrame(baris, columns=["pola", "dukungan", "p_beli"])
print("rata-rata p_beli seluruh sesi:", round(beli.mean(), 3))
print(tab.sort_values("p_beli").iloc[[0, 1, 2, -3, -2, -1]]
      .to_string(index=False, float_format="%.2f"))

## Listing 24.5: Memuat dan menyiapkan barisan sesi

`kode/bab24/siapkan.py`

In [ ]:
import numpy as np
import pandas as pd

KEJADIAN = ["beranda", "promo", "cari", "lihat_menu", "detail_menu",
            "keranjang", "bayar", "selesai", "keluar"]
SELESAI, KELUAR = 7, 8


def muat(path="kode/data/sesi_aplikasi.csv", batas="2026-03-22"):
    d = pd.read_csv(path, parse_dates=["waktu"])
    d["kode"] = d.kejadian.map({k: i for i, k in enumerate(KEJADIAN)})
    g = d.sort_values(["id_sesi", "urutan"]).groupby("id_sesi")
    seq = g.kode.apply(np.array)
    info = pd.DataFrame({"mulai": g.waktu.min(),
                         "panjang": g.size(),
                         "beli": seq.map(lambda s: s[-1] == SELESAI)})
    info["latih"] = info.mulai < batas      # dibagi menurut waktu
    niat = g.niat_laten.apply(np.array)     # hanya di data sintetis
    return seq, info, niat


def pad(daftar):
    """Barisan beragam panjang -> matriks O dan topeng M."""
    T = max(map(len, daftar))
    O = np.zeros((len(daftar), T), int)
    M = np.zeros((len(daftar), T), bool)
    for i, s in enumerate(daftar):
        O[i, :len(s)], M[i, :len(s)] = s, True
    return O, M

## Listing 24.6: Rantai Markov dan HMM biasa untuk sesi aplikasi

`kode/bab24/kasus_markov.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab24')
import warnings
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM
from siapkan import KEJADIAN, SELESAI, muat
warnings.simplefilter("ignore")

seq, info, _ = muat()
print(f"{len(seq)} sesi, {info.panjang.sum()} kejadian; "
      f"latih {info.latih.sum()}, uji {(~info.latih).sum()}")
print(f"proporsi beli {info.beli.mean():.3f}; median panjang "
      f"{info.panjang.median():.0f} kejadian")
latih, uji = seq[info.latih], seq[~info.latih]

V = len(KEJADIAN)
n = np.zeros((V, V))
for s in latih:
    np.add.at(n, (s[:-1], s[1:]), 1)
n[SELESAI, SELESAI] = n[SELESAI + 1, SELESAI + 1] = 1
P = n / n.sum(1, keepdims=True)
Q, R = P[:7, :7], P[:7, 7:]
N = np.linalg.inv(np.eye(7) - Q)
tab = pd.DataFrame({"p_keluar_langsung": P[:7, 8],
                    "p_beli_akhirnya": (N @ R)[:, 0],
                    "sisa_langkah": N.sum(1)}, index=KEJADIAN[:7])
print("\nRantai Markov (data latih):")
print(tab.round(2).to_string())


def ll_markov(daftar):
    return sum(np.log(P[s[:-1], s[1:]]).sum() for s in daftar)


print(f"\nlog-likelihood uji rantai Markov: {ll_markov(uji):.0f}")

X = np.concatenate(latih.values).reshape(-1, 1)
L = latih.map(len).to_numpy()
Xu = np.concatenate(uji.values).reshape(-1, 1)
Lu = uji.map(len).to_numpy()
hmm = {}
for S in (2, 3, 4):
    hmm[S] = CategoricalHMM(S, n_iter=100, tol=0.5, random_state=1,
                            n_features=V).fit(X, L)
    print(f"HMM biasa {S} keadaan: log-lik uji "
          f"{hmm[S].score(Xu, Lu):.0f}, BIC {hmm[S].bic(X, L):.0f}")

print("\nemisi tertinggi HMM biasa 3 keadaan:")
for j, b in enumerate(hmm[3].emissionprob_):
    top = np.argsort(b)[::-1][:3]
    print(f"  keadaan {j}: " + ", ".join(
        f"{KEJADIAN[k]} {b[k]:.2f}" for k in top))
print("  transisi:", hmm[3].transmat_.round(2).tolist())

## Listing 24.7: HMM beralih dengan NumPy

`kode/bab24/hmm_beralih.py`

In [ ]:
"""HMM dengan emisi Markov: P(o_t | o_t-1, z_t) = E[z_t, o_t-1, o_t].
Semua sesi diproses serentak (tervektorisasi) dengan penskalaan."""
import numpy as np


def maju(O, M, pi, A, E):
    n, T = O.shape
    a = np.zeros((n, T, len(pi)))
    c = np.ones((n, T))
    a[:, 0] = pi                     # kejadian pertama selalu beranda
    for t in range(1, T):
        x = (a[:, t - 1] @ A) * E[:, O[:, t - 1], O[:, t]].T
        m = M[:, t]
        c[m, t] = x[m].sum(1)
        a[m, t] = x[m] / c[m, t, None]
        a[~m, t] = a[~m, t - 1]      # sesi sudah berakhir
    return a, c


def mundur(O, M, A, E, c):
    n, T = O.shape
    b = np.ones((n, T, A.shape[0]))
    for t in range(T - 2, -1, -1):
        e = E[:, O[:, t], O[:, t + 1]].T
        x = ((e * b[:, t + 1]) @ A.T) / c[:, t + 1, None]
        m = M[:, t + 1]
        b[m, t] = x[m]
    return b


def latih(O, M, S=2, iterasi=60, seed=0, V=9):
    r = np.random.default_rng(seed)
    pi = r.dirichlet(np.ones(S))
    A = np.full((S, S), 0.1 / (S - 1))
    np.fill_diagonal(A, 0.9)
    E = r.dirichlet(np.ones(V), (S, V))
    for _ in range(iterasi):
        a, c = maju(O, M, pi, A, E)                              # E
        b = mundur(O, M, A, E, c)
        g = a * b
        An = np.zeros((S, S))
        En = np.full((S, V, V), 1e-3)     # penghalusan kecil
        for t in range(1, O.shape[1]):                           # M
            m = M[:, t]
            e = E[:, O[:, t - 1], O[:, t]].T
            xi = (a[m, t - 1, :, None] * A
                  * (e[m] * b[m, t])[:, None, :])
            xi /= c[m, t, None, None]
            An += xi.sum(0)
            for s in range(S):
                np.add.at(En[s], (O[m, t - 1], O[m, t]), g[m, t, s])
        pi = g[:, 0].mean(0)
        A = An / An.sum(1, keepdims=True)
        E = En / En.sum(2, keepdims=True)
    loglik = np.log(maju(O, M, pi, A, E)[1]).sum()
    return pi, A, E, loglik

## Listing 24.8: Melatih dan mengevaluasi HMM beralih

`kode/bab24/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab24')
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from siapkan import KEJADIAN, SELESAI, muat, pad
from hmm_beralih import latih, maju, mundur

seq, info, niat = muat()
O, M = pad(list(seq[info.latih]))
Ou, Mu = pad(list(seq[~info.latih]))

hasil = [latih(O, M, S=2, seed=s) for s in range(3)]
for s, h in enumerate(hasil):
    print(f"awal acak {s}: log-lik latih {h[3]:.0f}")
pi, A, E, _ = max(hasil, key=lambda h: h[3])
ll_uji = np.log(maju(Ou, Mu, pi, A, E)[1]).sum()
print(f"log-lik uji HMM beralih: {ll_uji:.0f}")

nt = int(np.argmax(E[:, 5, 6]))     # keranjang -> bayar
jl = 1 - nt
print(f"\nkeadaan niat = {nt}; pi(niat) = {pi[nt]:.2f}")
print(f"peluang tetap: jelajah {A[jl, jl]:.3f}, "
      f"niat {A[nt, nt]:.3f}")
K = {k: i for i, k in enumerate(KEJADIAN)}
for a, b in [("beranda", "promo"), ("beranda", "cari"),
             ("lihat_menu", "detail_menu"), ("detail_menu",
             "keranjang"), ("keranjang", "bayar"),
             ("keranjang", "keluar"), ("bayar", "selesai")]:
    print(f"  {a:>11s} -> {b:<11s} jelajah {E[jl, K[a], K[b]]:.2f}"
          f"   niat {E[nt, K[a], K[b]]:.2f}")

a, c = maju(Ou, Mu, pi, A, E)
g = a * mundur(Ou, Mu, A, E, c)
benar = np.zeros_like(Ou)
for i, z in enumerate(niat[~info.latih]):
    benar[i, :len(z)] = z
for nama, p in [("penyaringan", a), ("penghalusan", g)]:
    tepat = ((p[..., nt] > .5) == benar)[Mu].mean()
    print(f"kecocokan {nama} dengan niat_laten: {tepat:.3f}")


def serap(P):
    P = P.copy()
    P[SELESAI] = P[SELESAI + 1] = 0
    P[SELESAI, SELESAI] = P[SELESAI + 1, SELESAI + 1] = 1
    N = np.linalg.inv(np.eye(7) - P[:7, :7])
    return (N @ P[:7, 7:])[:, 0]


n = np.zeros((9, 9))
for s in seq[info.latih]:
    np.add.at(n, (s[:-1], s[1:]), 1)
n[7:, :] += np.eye(9)[7:]
b_markov = serap(n / n.sum(1, keepdims=True))
b_rezim = np.array([serap(E[j]) for j in range(2)])


def ciri(s, k):
    hit = np.bincount(s[:k], minlength=9)[:7]
    return np.r_[hit, np.eye(7)[s[k - 1]]]


print("\nAUC menebak pembelian setelah k kejadian (data uji):")
print("  k  sesi  beli  halaman  logistik  HMM-niat  HMM-beli")
uji, lat = seq[~info.latih], seq[info.latih]
auc = []
for k in (2, 3, 4, 5, 6):
    idx = [i for i, s in enumerate(uji) if len(s) > k
           and s[k - 1] < SELESAI]
    y = np.array([uji.iloc[i][-1] == SELESAI for i in idx])
    hal = b_markov[[uji.iloc[i][k - 1] for i in idx]]
    tr = [s for s in lat if len(s) > k and s[k - 1] < SELESAI]
    lr = LogisticRegression(max_iter=2000).fit(
        [ciri(s, k) for s in tr], [s[-1] == SELESAI for s in tr])
    plr = lr.predict_proba([ciri(uji.iloc[i], k) for i in idx])[:, 1]
    pn = a[idx, k - 1, nt]
    pb = (a[idx, k - 1] * b_rezim[:, Ou[idx, k - 1]].T).sum(1)
    baris = [roc_auc_score(y, p) for p in (hal, plr, pn, pb)]
    auc.append([k] + baris)
    print(f"  {k}  {len(idx):4d}  {y.mean():.2f}  " +
          "  ".join(f"{v:7.3f}" for v in baris))
pd.DataFrame(auc, columns=["k", "hal", "lr", "niat", "beli"]).to_csv(
    "gambar/bab24/auc.dat", sep=" ", index=False, float_format="%.3f")
np.savez("kode/bab24/_model.npz", pi=pi, A=A, E=E, nt=nt,
         b_markov=b_markov)